In [ ]:
!pip install -q llama-index llama-index-llms-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 9.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

In [ ]:
!pip install -q pymupdf llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 42.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
!pip install -q llama-index-retrievers-bm25 sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.3/683.3 kB 9.7 MB/s eta 0:00:00


In [ ]:
!pip install -q llama-index llama-index-llms-groq pymupdf llama-index-embeddings-huggingface llama-index-retrievers-bm25 sentence-transformers
import fitz
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get('Groq')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("API key accepted!")

from google.colab import files

print("Please select your PDF file to upload:")
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]

import os
os.makedirs("sample_docs", exist_ok=True)
pdf_path = os.path.join("sample_docs", pdf_filename)

with open(pdf_path, 'wb') as f:
  f.write(uploaded[pdf_filename])


print(f"PDF saved to {pdf_path}")

API key accepted!
Please select your PDF file to upload:


Saving sample-sdf-document.pdf to sample-sdf-document.pdf
PDF saved to sample_docs/sample-sdf-document.pdf


In [ ]:
doc = fitz.open(pdf_path)
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the PDF.")

Extracted 617 words from the PDF.


In [ ]:
from llama_index.llms.groq import Groq
from llama_index.core.llms import ChatMessage

llm = Groq(model="llama-3.1-8b-instant")

def rewrite_query(user_query):
  messages = [
      ChatMessage(
          role="system",
          content="You are helping improve search queries for a pharmaceutical quality document (a Certificate of Quality / SDF). Rewrite the user's query to also include related pharmaceutical and quality control terminology that might appear in such a document. Only output the rewritten query and nothing else."
      ),
      ChatMessage(role="user", content=user_query),
  ]
  response = llm.chat(messages)
  return response.message.content

query = "What are the storage conditions for this product?"
additional_query = rewrite_query(query)

print(f"Original Query: {query}")
print(f"Additional Query: {additional_query}")

Original Query: What are the storage conditions for this product?
Additional Query: What are the storage conditions specified in the Certificate of Quality (CoQ) or Stability Data Package (SDP) for this pharmaceutical product, including temperature, humidity, and light exposure requirements, as well as any additional storage conditions such as refrigeration or freezing?


In [ ]:
from llama_index.core import Document, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

doc_pages = fitz.open(pdf_path)
documents = []

for i, page in enumerate(doc_pages):
  page_text = page.get_text()
  if page_text.strip():
    documents.append(
        Document(
            text=page_text,
            metadata={"page_number": i + 1}
        )
    )

doc_pages.close()

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

print(f"Indexed {len(documents)} pages successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 3 pages successfully.


In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever

def create_hybrid_retriever(index, query, top_k=2):
  vector_retriever = index.as_retriever(similarity_top_k=top_k)
  vector_nodes = vector_retriever.retrieve(query)

  nodes = [node for node in index.docstore.docs.values()]
  bm25_retriever = BM25Retriever.from_defaults(
      nodes=nodes,
      similarity_top_k=top_k
  )
  keyword_nodes = bm25_retriever.retrieve(query)

  all_nodes = list(vector_nodes) + list(keyword_nodes)

  unique_nodes = {}
  for node in all_nodes:
    if node.node_id not in unique_nodes:
      unique_nodes[node.node_id] = node


  sorted_nodes = sorted(
        unique_nodes.values(),
        key=lambda x: x.score if hasattr(x, 'score') else 0.0,
        reverse=True
    )
  return sorted_nodes[:top_k]


hybrid_nodes = create_hybrid_retriever(index, "What are the storage conditions for this product?")

for i, node in enumerate(hybrid_nodes):
  print(f"Result {i+1} (Score: {node.score:.4f})")
  print(node.get_text()[:200])
  print("-" * 40)

DEBUG:bm25s:Building index from IDs objects


Result 1 (Score: 0.2206)
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature f
----------------------------------------
Result 2 (Score: 0.1168)
Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quality management system.
Issued by Cytiva Westborough Quality Assurance
This document has been electron
----------------------------------------


In [ ]:
response2_fixed = query_engine.query(additional_query)
print(response2_fixed)

NameError: name 'query_engine' is not defined

In [ ]:
hybrid_nodes_expanded = create_hybrid_retriever(index, additional_query, top_k=2)
print("Results using EXPANDED query:")
for i, node in enumerate(hybrid_nodes_expanded):
    print(f"Result {i+1} (Score: {node.score:.4f}, Page: {node.metadata.get('page_number')}):")
    print(node.get_text()[:150])
    print("-" * 40)

DEBUG:bm25s:Building index from IDs objects


Results using EXPANDED query:
Result 1 (Score: 0.3136, Page: 1):
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It 
----------------------------------------
Result 2 (Score: 0.2753, Page: 3):
Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 9001 certified quality management system.
Issued by C
----------------------------------------


In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank

reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=2
)


retriever = index.as_retriever(similarity_top_k=3)
query = "What are the storage conditions for this product?"
retrieved_nodes = retriever.retrieve(query)

print("Before Reranking:")
for i, node in enumerate(retrieved_nodes):
  print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")

reranked_nodes = reranker.postprocess_nodes(
    retrieved_nodes,
    query_str=query
)

print("\nAfter Reranking:")
for i, node in enumerate(reranked_nodes):
  print(f"{i+1}.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Before Reranking:
1. (Score: 0.2206) - Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄK...
2. (Score: 0.1168) - Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quali...
3. (Score: 0.0982) - Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 900...

After Reranking:
1.
2.


In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import BaseRetriever

class HybridRetriever(BaseRetriever):
    def __init__(self, vector_retriever, keyword_retriever, top_k=2):
        self.vector_retriever = vector_retriever
        self.keyword_retriever = keyword_retriever
        self.top_k = top_k
        super().__init__()

    def _retrieve(self, query_bundle, **kwargs):
        vector_nodes = self.vector_retriever.retrieve(query_bundle)
        keyword_nodes = self.keyword_retriever.retrieve(query_bundle)

        all_nodes = list(vector_nodes) + list(keyword_nodes)
        unique_nodes = {}
        for node in all_nodes:
            if node.node_id not in unique_nodes:
                unique_nodes[node.node_id] = node

        sorted_nodes = sorted(
            unique_nodes.values(),
            key=lambda x: x.score if hasattr(x, 'score') else 0.0,
            reverse=True
        )
        return sorted_nodes[:self.top_k]

nodes = list(index.docstore.docs.values())
vector_retriever = index.as_retriever(similarity_top_k=2)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=2)
hybrid_retriever = HybridRetriever(vector_retriever, bm25_retriever, top_k=2)

query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,
    llm=llm,
    node_postprocessors=[reranker]
)

print("Query engine built successfully!")

DEBUG:bm25s:Building index from IDs objects


Query engine built successfully!


In [ ]:
response2_fixed = query_engine.query(additional_query)
print(response2_fixed)

The storage conditions for this pharmaceutical product are not explicitly mentioned in the provided Certificate of Quality or any other relevant documents. However, it is recommended to store the product at a temperature above +5 C to prevent brittleness or cracking of the plastic connectors. The operating temperature of the product is specified as +2 C to +40 C.


In [ ]:
print(f"Number of retrieved nodes: {len(retrieved_nodes)}")
print(f"Number of reranked nodes: {len(reranked_nodes)}")

print("\nReranked nodes content: ")
for node in reranked_nodes:
  print(node)
  print("---")

Number of retrieved nodes: 3
Number of reranked nodes: 2

Reranked nodes content: 
Node ID: 486ea637-eeb0-42b0-8915-e96e25a4410c
Text: Cytiva 100 Results Way Marlborough, MA 01752 United States Page
1 / 1 cytiva.com 3 June, 2022 Re: ÄKTATM ready Flow Kit Storage
Conditions To Whom It May Concern, The recommended storage temperature
for standard ÄKTA ready flow kits is provided in Section 8.3 of the
Operating  Instructions 28960345 and specified as > +5 C. This
recommendation al...
Score: -1.104

---
Node ID: 0658a9ab-b7c6-4b05-9a2e-d4eaa2df648d
Text: Certificate of Quality  This product is manufactured in
compliance with our ISO 9001 certified quality management system.
Issued by Cytiva Westborough Quality Assurance This document has been
electronically produced and is valid without a signature. ÄKTA is a
trademark of Global Life Sciences Solutions USA LLC or an affiliate
doing business as C...
Score: -9.482

---


In [ ]:
retriever = index.as_retriever(similarity_top_k=3)
query = "What are the storage conditions for this product?"
retrieved_nodes = retriever.retrieve(query)

print(f"Retrieved {len(retrieved_nodes)} nodes")
for i, node in enumerate(retrieved_nodes):
  print(f"{i+1}. Score: {node.score:.4f} | Page: {node.metadata.get('page_number')}")


Retrieved 3 nodes
1. Score: 0.2206 | Page: 1
2. Score: 0.1168 | Page: 2
3. Score: 0.0982 | Page: 3


In [ ]:
print(f"Number of documents created: {len(documents)}")
for i, d in enumerate(documents):
  print(f"\nDocument {i+1} (page {d.metadata.get('page_number')}):")
  print(f"Length: {len(d.text)} characters")
  print(d.text[:100])

Number of documents created: 3

Document 1 (page 1):
Length: 1281 characters
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄK

Document 2 (page 2):
Length: 1364 characters
Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quali

Document 3 (page 3):
Length: 1378 characters
Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 900
